### 7.1 Dataset

In [10]:
import pandas as pd

df = pd.read_json('c4-train.00000-of-01024-30K.json', lines = True)
df.head()

,text,timestamp,url
0,Beginners BBQ Class Taking Place in Missoula!\...,2019-04-25 12:57:54+00:00,https://klyq.com/beginners-bbq-class-taking-pl...
1,Discussion in 'Mac OS X Lion (10.7)' started b...,2019-04-21 10:07:13+00:00,https://forums.macrumors.com/threads/restore-f...
2,Foil plaid lycra and spandex shortall with met...,2019-04-25 10:40:23+00:00,https://awishcometrue.com/Catalogs/Clearance/T...
3,How many backlinks per day for new site?\nDisc...,2019-04-21 12:46:19+00:00,https://www.blackhatworld.com/seo/how-many-bac...
4,The Denver Board of Education opened the 2017-...,2019-04-20 14:33:21+00:00,http://bond.dpsk12.org/category/news/


### 7.2

In [11]:
df['clean_text'] = (
    df['text']
    .fillna("")
    .str.lower()
    .str.replace(r'[^a-z0-9\s]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
df.head()

,text,timestamp,url,clean_text
0,Beginners BBQ Class Taking Place in Missoula!\...,2019-04-25 12:57:54+00:00,https://klyq.com/beginners-bbq-class-taking-pl...,beginners bbq class taking place in missoula d...
1,Discussion in 'Mac OS X Lion (10.7)' started b...,2019-04-21 10:07:13+00:00,https://forums.macrumors.com/threads/restore-f...,discussion in mac os x lion 10 7 started by ax...
2,Foil plaid lycra and spandex shortall with met...,2019-04-25 10:40:23+00:00,https://awishcometrue.com/Catalogs/Clearance/T...,foil plaid lycra and spandex shortall with met...
3,How many backlinks per day for new site?\nDisc...,2019-04-21 12:46:19+00:00,https://www.blackhatworld.com/seo/how-many-bac...,how many backlinks per day for new site discus...
4,The Denver Board of Education opened the 2017-...,2019-04-20 14:33:21+00:00,http://bond.dpsk12.org/category/news/,the denver board of education opened the 2017 ...


### 7.3

In [19]:
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse as sp
import numpy as np

# --- 1. Raw documents & 2. Tokenizer & 3. CountVectorizer ---
vectorizer = CountVectorizer()
count_matrix = vectorizer.fit_transform(df['clean_text'])


row_sums = np.array(count_matrix.sum(axis=1)).ravel()

row_sums[row_sums == 0] = 1


TF = count_matrix.multiply(1 / row_sums[:, np.newaxis])
TF = sp.csr_matrix(TF)


N, V = count_matrix.shape
df_counts = np.bincount(count_matrix.indices, minlength=V)
idf_scores = np.log((1 + N) / (1 + df_counts)) + 1


IDF_diagonal = sp.diags(idf_scores)
X = TF.dot(IDF_diagonal)

print(f"Number of documents (N) : {N:,}")
print(f"Vocabulary size (V)      : {V:,}")
print(f"Ma trận TF-IDF đã được tạo thành công với kích thước: {X.shape}")

Number of documents (N) : 30,000
Vocabulary size (V)      : 186,582
Ma trận TF-IDF đã được tạo thành công với kích thước: (30000, 186582)


### 7.4 Kiểm tra Sparsity
Tính độ thưa của ma trận:
$$S = 1 - \frac{nnz(X)}{N \times V}$$
trong đó $nnz(X)$ là số phần tử khác 0.

In [20]:
# 7.4 Kiem tra sparsity
nnz = X.nnz
total_elements = N * V
sparsity = 1.0 - (nnz / total_elements)

print(f"So phan tu khac 0 (nnz)  : {nnz:,}")
print(f"Tong so phan tu (N x V)  : {total_elements:,}")
print(f"Matrix Sparsity (S)      : {sparsity * 100:.4f}%")

So phan tu khac 0 (nnz)  : 4,981,696
Tong so phan tu (N x V)  : 5,597,460,000
Matrix Sparsity (S)      : 99.9110%


****Trả lời câu hỏi 7.4:****

**Tại sao một document chỉ sử dụng một phần rất nhỏ vocabulary nhưng vector vẫn có chiều \(V\)?**

- ****Trả lời:**** Trong mô hình không gian vector (Vector Space Model), toàn bộ document được biểu diễn dựa trên ****một vocabulary chung có \(V\) term****. Mỗi term trong vocabulary tương ứng với một chiều cố định của vector. Vì vậy, dù một document chỉ chứa một số ít term, vector của nó vẫn phải có đủ **\(V\) chiều** để đảm bảo tất cả các document đều nằm trong cùng một không gian và có thể so sánh với nhau. Các term không xuất hiện trong document sẽ có giá trị **\(0\)** ở những chiều tương ứng.


### 7.5 Inspect Vocabulary
Tìm:
1. 20 terms phổ biến nhất theo document frequency (DF).
2. 20 terms có IDF cao nhất.
3. 20 terms có TF-IDF cao nhất trong một document được chọn (Doc 0).

In [23]:
import numpy as np
import pandas as pd

feature_names = np.array(vectorizer.get_feature_names_out())

# Tính Document Frequency thực tế bằng cách đếm số document chứa mỗi term
df_counts = np.bincount(X.indices, minlength=V)

# Tính toán IDF theo công thức chuẩn của scikit-learn: ln((1 + N) / (1 + df)) + 1
idf_scores = np.log((1 + N) / (1 + df_counts)) + 1

# 1. 20 terms phổ biến nhất theo document frequency
top20_df_idx = np.argsort(df_counts)[::-1][:20]
df_top20_df = pd.DataFrame({
    "Rank": range(1, 21),
    "Term": feature_names[top20_df_idx],
    "Doc Frequency (df)": df_counts[top20_df_idx],
    "IDF": np.round(idf_scores[top20_df_idx], 4)
})

# 2. 20 terms có IDF cao nhất
top20_idf_idx = np.argsort(idf_scores)[::-1][:20]
df_top20_idf = pd.DataFrame({
    "Rank": range(1, 21),
    "Term": feature_names[top20_idf_idx],
    "Doc Frequency (df)": df_counts[top20_idf_idx],
    "IDF": np.round(idf_scores[top20_idf_idx], 4)
})

# 3. 20 terms có tần suất cao nhất trong Document 0
doc0_vec = X[0].toarray().ravel()
top20_tf_idf = np.argsort(doc0_vec)[::-1][:20]
df_top20_doc0 = pd.DataFrame({
    "Rank": range(1, 21),
    "Term": feature_names[top20_tf_idf],
    "Term Frequency (TF)": doc0_vec[top20_tf_idf],
    "IDF": np.round(idf_scores[top20_tf_idf], 4)
})

print("--- TOP 20 TERMS PHO BIEN NHAT (THEO DF) ---")
print(df_top20_df.head(20).to_string(index=False))

print("\n--- TOP 20 TERMS CO IDF CAO NHAT ---")
print(df_top20_idf.head(20).to_string(index=False))

print(f"\n--- TOP 20 TERMS CO TFIDF CAO NHAT TRONG DOC 0 ---")
print(df_top20_doc0.head(20).to_string(index=False))

--- TOP 20 TERMS PHO BIEN NHAT (THEO DF) ---
 Rank Term  Doc Frequency (df)    IDF
    1  the               27893 1.0728
    2  and               27423 1.0898
    3   to               26689 1.1169
    4   of               26031 1.1419
    5   in               25224 1.1734
    6  for               23651 1.2378
    7   is               22740 1.2771
    8 with               21405 1.3376
    9   on               20265 1.3923
   10 that               18370 1.4905
   11 this               17841 1.5197
   12  are               17594 1.5336
   13   it               17174 1.5578
   14   as               16473 1.5994
   15   at               16348 1.6071
   16 from               16316 1.6090
   17   be               16154 1.6190
   18  you               16095 1.6227
   19   by               15123 1.6849
   20 have               14852 1.7030

--- TOP 20 TERMS CO IDF CAO NHAT ---
 Rank       Term  Doc Frequency (df)     IDF
    1      00000                   1 10.6158
    2 zzzzzzz333             

**Câu hỏi 1:** *Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không?*

- **Trả lời:** **Không.** Một term xuất hiện trong càng nhiều document thì $df(t)$ càng lớn, làm cho $idf(t)=\log\frac{N}{df(t)}$ giảm xuống. Nếu term xuất hiện trong toàn bộ corpus thì $df(t)=N$, khi đó $idf(t)=0$. Vì vậy, dù $tf(t,d)$ lớn do term xuất hiện nhiều lần trong document, giá trị TF-IDF vẫn rất thấp hoặc bằng 0.

**Câu hỏi 2:** *Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?*

- **Trả lời:** **Không.** IDF cao cho thấy term đó chỉ xuất hiện ở một số ít document trong corpus. Tuy nhiên, TF-IDF còn phụ thuộc vào $TF$ trong từng document. Nếu document không chứa term đó thì $tf(t,d)=0$, do đó $tfidf(t,d)=0\times idf=0$. Vì vậy, IDF cao chỉ góp phần tạo ra TF-IDF cao ở các document có chứa term đó, đặc biệt khi term xuất hiện với tần suất tương đối lớn.